## 1: Algoritmo cliente

Paso 1: Definicion de clases (algoritmo de distribución y métricas)

In [5]:
# clase metricas

import numpy as np
from collections import defaultdict

class TeamMetrics:
    def __init__(self, assignments, student_assignments, students, challenges):
        self.assignments = assignments
        self.student_assignments = student_assignments
        self.students = students
        self.challenges = challenges

    def calculate_student_satisfaction(self):
        """
        Calcula la satisfacción promedio de los estudiantes.
        Satisfacción = 1/(índice_preferencia + 1)
        """
        total_satisfaction = 0
        n_students = len(self.students)

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge in student['Postulaciones']:
                preference_index = student['Postulaciones'].index(assigned_challenge)
                satisfaction = 1 / (preference_index + 1)
            else:
                satisfaction = 0
            total_satisfaction += satisfaction

        return total_satisfaction / n_students

    def count_first_priority_assignments(self):
        """
        Cuenta cuántos estudiantes quedaron en su primera preferencia
        """
        first_priority_count = 0

        for student in self.students:
            if (len(student['Postulaciones']) > 0 and
                self.student_assignments.get(student['Nombre']) == student['Postulaciones'][0]):
                first_priority_count += 1

        return first_priority_count

    def count_outside_preferences(self):
        """
        Cuenta cuántos estudiantes quedaron fuera de sus preferencias
        """
        outside_preferences_count = 0

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge not in student['Postulaciones']:
                outside_preferences_count += 1

        return outside_preferences_count

    def count_challenges_without_team(self):
        """
        Cuenta cuántos desafíos quedaron sin equipo (menos de 2 estudiantes)
        """
        challenges_without_team = 0
        challenges_with_students = set(self.assignments.keys())

        # Contar desafíos sin equipo entre los que tienen al menos un estudiante
        for challenge in challenges_with_students:
            if len(self.assignments[challenge]) < 2:
                challenges_without_team += 1

        # Agregar desafíos que no tienen ningún estudiante
        all_challenge_titles = {challenge['Titulo'] for challenge in self.challenges}
        challenges_without_team += len(all_challenge_titles - challenges_with_students)

        return challenges_without_team

    def calculate_team_size_std(self):
        """
        Calcula la desviación estándar del tamaño de los equipos que tienen estudiantes
        """
        team_sizes = [len(team) for team in self.assignments.values() if len(team) > 0]
        return np.std(team_sizes) if team_sizes else 0

    def calculate_average_careers_per_team(self):
        """
        Calcula el promedio de carreras diferentes por equipo
        """
        careers_per_team = []

        for team in self.assignments.values():
            if len(team) >= 2:  # Solo considerar equipos válidos
                unique_careers = len(set(student['Carrera'] for student in team))
                careers_per_team.append(unique_careers)

        return np.mean(careers_per_team) if careers_per_team else 0

    def get_all_metrics(self):
        """
        Calcula y retorna todas las métricas en un diccionario
        """
        metrics = {
            'satisfaccion_promedio': self.calculate_student_satisfaction(),
            'estudiantes_primera_prioridad': self.count_first_priority_assignments(),
            'estudiantes_fuera_preferencias': self.count_outside_preferences(),
            'desafios_sin_equipo': self.count_challenges_without_team(),
            'std_tamaño_equipos': self.calculate_team_size_std(),
            'promedio_carreras_por_equipo': self.calculate_average_careers_per_team()
        }

        return metrics

    def print_metrics(self):
        """
        Imprime todas las métricas de forma legible
        """
        metrics = self.get_all_metrics()

        print("\n=== Métricas de Asignación ===")
        print(f"Satisfacción promedio: {metrics['satisfaccion_promedio']:.3f}")
        print(f"Estudiantes en primera prioridad: {metrics['estudiantes_primera_prioridad']}")
        print(f"Estudiantes fuera de preferencias: {metrics['estudiantes_fuera_preferencias']}")
        print(f"Desafíos sin equipo: {metrics['desafios_sin_equipo']}")
        print(f"Desviación estándar tamaño equipos: {metrics['std_tamaño_equipos']:.3f}")
        print(f"Promedio de carreras por equipo: {metrics['promedio_carreras_por_equipo']:.2f}")

        # Estadísticas adicionales
        total_students = len(self.students)
        print(f"\nPorcentajes:")
        print(f"Primera prioridad: {(metrics['estudiantes_primera_prioridad']/total_students)*100:.1f}%")
        print(f"Fuera de preferencias: {(metrics['estudiantes_fuera_preferencias']/total_students)*100:.1f}%")

In [2]:
# clase algoritmo

import random
from collections import defaultdict
from typing import List, Dict, Set, Tuple

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = self.students.copy()
        
    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student in self.unassigned_students:
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1
    
    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True
    
    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student in self.unassigned_students:
            if challenge in student['Postulaciones']:
                # Remover el desafío completado
                student['Postulaciones'].remove(challenge)
    
    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.remove(student)
    
    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            # Contar postulaciones por desafío
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                # Asignar estudiantes restantes a desafíos random que cumplan restricciones
                self.assign_remaining_students()
                break
                
            # Obtener desafío con más postulaciones
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            # Obtener estudiantes que tienen este desafío como primera prioridad
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            # Seleccionar estudiantes aleatoriamente cumpliendo restricciones
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
                        
            # Actualizar prioridades de estudiantes no seleccionados
            self.update_student_priorities(current_challenge)
            
        return self.assignments, self.student_assignments
    
    def assign_remaining_students(self):
        """Asigna los estudiantes restantes a desafíos que cumplan las restricciones."""
        for student in self.unassigned_students[:]:
            for challenge in [c['Titulo'] for c in self.challenges]:
                if self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    break
    
    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        for challenge, team in self.assignments.items():
            # Verificar tamaño del equipo
            if not (2 <= len(team) < 5):
                print(f"Error: Equipo de tamaño inválido en {challenge}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Verificar que todos los estudiantes estén asignados
        if self.unassigned_students:
            print("Error: Hay estudiantes sin asignar")
            return False
            
        return True
    
    def print_assignment_stats(self):
        print("\n=== Estadísticas de Asignación de TeamFormationHeuristic ===\n")
        for challenge, team in self.assignments.items():
            print(f"Desafío: {challenge}")
            print(f"Número de estudiantes asignados: {len(team)}")
            
            # Distribución por carrera
            career_distribution = defaultdict(int)
            for student_id in team:
                student_data = self.student_data[student_id]
                career = student_data['Carrera']
                career_distribution[career] += 1
                
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = self.career_limits.get(career, "No especificado")
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            # Listado de estudiantes en el equipo
            print("\nEstudiantes en el equipo:")
            for student_id in team:
                student_data = self.student_data[student_id]
                nombre = student_data['Nombre']
                carrera = student_data['Carrera']
                preferencias = student_data['Postulaciones']
                
                try:
                    preferencia_numero = preferencias.index(challenge) + 1
                    preferencia = f"(Preferencia #{preferencia_numero})"
                except ValueError:
                    preferencia = "(No preferido)"
                print(f"- {nombre} ({carrera}) {preferencia}")
            
            print("\n")  # Espacio entre desafíos
        print("✅ Resumen de asignación completado.\n")


def main(data):
    heuristic = TeamFormationHeuristic(data)
    assignments, student_assignments = heuristic.form_teams()
    
    if heuristic.validate_assignments():
        print("Asignaciones válidas completadas")
        
        # Crear instancia de TeamMetrics y calcular métricas
        metrics = TeamMetrics(
            assignments=assignments,
            student_assignments=student_assignments,
            students=data['estudiantes'],
            challenges=data['desafios']
        )
        
        # Imprimir métricas
        metrics.print_metrics()
        
        # Imprimir asignaciones
        print("\nAsignaciones finales:")
        for challenge, team in assignments.items():
            print(f"\nDesafío: {challenge}")
            print("Equipo:")
            for student in team:
                print(f"- {student['Nombre']} ({student['Carrera']})")
    else:
        print("Error en las asignaciones")
    
    return assignments, student_assignments

Paso 2: Imprimir resultados

In [8]:
# Cargar datos
#data = ... # Tu JSON de entrada
import json

with open('body.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Crear instancia y formar equipos
heuristic = TeamFormationHeuristic(data)
assignments, student_assignments = heuristic.form_teams()

# Calcular métricas
metrics = TeamMetrics(
    assignments=assignments,
    student_assignments=student_assignments,
    students=data['estudiantes'],
    challenges=data['desafios']
)

# Imprimir métricas
heuristic.print_assignment_stats()
metrics.print_metrics()


=== Estadísticas de Asignación de TeamFormationHeuristic ===

Desafío: Clasificación de imágenes de mamografía usando Machine Learning
Número de estudiantes asignados: 3


AttributeError: 'TeamFormationHeuristic' object has no attribute 'student_data'

In [ ]:
import random
from collections import defaultdict
from typing import List, Dict, Set, Tuple

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = self.students.copy()
        self.assignment_order = []  # Lista para mantener el orden de asignación
        
    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student in self.unassigned_students:
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1
    
    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True
    
    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student in self.unassigned_students:
            if challenge in student['Postulaciones']:
                # Remover el desafío completado
                student['Postulaciones'].remove(challenge)
    
    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.remove(student)
    # [Métodos anteriores se mantienen igual hasta form_teams]
    
    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)  # Registrar orden de asignación
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.remove(student)
    
    def get_student_preference_number(self, student: dict, challenge: str) -> str:
        """Obtiene el número de preferencia de un estudiante para un desafío."""
        original_student = next(s for s in self.students if s['Nombre'] == student['Nombre'])
        if challenge in original_student['Postulaciones']:
            return f"Preferencia #{original_student['Postulaciones'].index(challenge) + 1}"
        return "Fuera de preferencias"
    
    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
            if len(team) < 2:  # Saltar equipos incompletos
                continue
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            # Calcular distribución por carrera
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_student_preference_number(student, challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")
    
    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                self.assign_remaining_students()
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
                        
            self.update_student_priorities(current_challenge)
            
        return self.assignments, self.student_assignments

def main(data):
    heuristic = TeamFormationHeuristic(data)
    assignments, student_assignments = heuristic.form_teams()
    
    if heuristic.validate_assignments():
        print("Asignaciones válidas completadas")
        
        # Crear instancia de TeamMetrics y calcular métricas
        metrics = TeamMetrics(
            assignments=assignments,
            student_assignments=student_assignments,
            students=data['estudiantes'],
            challenges=data['desafios']
        )
        
        # Imprimir métricas
        metrics.print_metrics()
        
        # Imprimir estadísticas detalladas de los equipos
        heuristic.print_team_statistics()
        
    else:
        print("Error en las asignaciones")
    
    return assignments, student_assignments

In [43]:
import random
from collections import defaultdict
from typing import List, Dict, Set, Tuple

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = self.students.copy()
        self.assignment_order = []  # Lista para mantener el orden de asignación
        
    # [Métodos anteriores se mantienen igual hasta form_teams]
    
    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)  # Registrar orden de asignación
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.remove(student)
    
    def get_student_preference_number(self, student: dict, challenge: str) -> str:
        """Obtiene el número de preferencia de un estudiante para un desafío."""
        original_student = next(s for s in self.students if s['Nombre'] == student['Nombre'])
        if challenge in original_student['Postulaciones']:
            return f"Preferencia #{original_student['Postulaciones'].index(challenge) + 1}"
        return "Fuera de preferencias"
    
    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
            if len(team) < 2:  # Saltar equipos incompletos
                continue
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            # Calcular distribución por carrera
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_student_preference_number(student, challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")
    
    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                self.assign_remaining_students()
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
                        
            self.update_student_priorities(current_challenge)
            
        return self.assignments, self.student_assignments

def main(data):
    heuristic = TeamFormationHeuristic(data)
    assignments, student_assignments = heuristic.form_teams()
    
    if heuristic.validate_assignments():
        print("Asignaciones válidas completadas")
        
        # Crear instancia de TeamMetrics y calcular métricas
        metrics = TeamMetrics(
            assignments=assignments,
            student_assignments=student_assignments,
            students=data['estudiantes'],
            challenges=data['desafios']
        )
        
        # Imprimir métricas
        metrics.print_metrics()
        
        # Imprimir estadísticas detalladas de los equipos
        heuristic.print_team_statistics()
        
    else:
        print("Error en las asignaciones")
    
    return assignments, student_assignments

In [44]:
# clase metricas

import numpy as np
from collections import defaultdict

class TeamMetrics:
    def __init__(self, assignments, student_assignments, students, challenges):
        self.assignments = assignments
        self.student_assignments = student_assignments
        self.students = students
        self.challenges = challenges

    def calculate_student_satisfaction(self):
        """
        Calcula la satisfacción promedio de los estudiantes.
        Satisfacción = 1/(índice_preferencia + 1)
        """
        total_satisfaction = 0
        n_students = len(self.students)

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge in student['Postulaciones']:
                preference_index = student['Postulaciones'].index(assigned_challenge)
                satisfaction = 1 / (preference_index + 1)
            else:
                satisfaction = 0
            total_satisfaction += satisfaction

        return total_satisfaction / n_students

    def count_first_priority_assignments(self):
        """
        Cuenta cuántos estudiantes quedaron en su primera preferencia
        """
        first_priority_count = 0

        for student in self.students:
            if (len(student['Postulaciones']) > 0 and
                self.student_assignments.get(student['Nombre']) == student['Postulaciones'][0]):
                first_priority_count += 1

        return first_priority_count

    def count_outside_preferences(self):
        """
        Cuenta cuántos estudiantes quedaron fuera de sus preferencias
        """
        outside_preferences_count = 0

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge not in student['Postulaciones']:
                outside_preferences_count += 1

        return outside_preferences_count

    def count_challenges_without_team(self):
        """
        Cuenta cuántos desafíos quedaron sin equipo (menos de 2 estudiantes)
        """
        challenges_without_team = 0
        challenges_with_students = set(self.assignments.keys())

        # Contar desafíos sin equipo entre los que tienen al menos un estudiante
        for challenge in challenges_with_students:
            if len(self.assignments[challenge]) < 2:
                challenges_without_team += 1

        # Agregar desafíos que no tienen ningún estudiante
        all_challenge_titles = {challenge['Titulo'] for challenge in self.challenges}
        challenges_without_team += len(all_challenge_titles - challenges_with_students)

        return challenges_without_team

    def calculate_team_size_std(self):
        """
        Calcula la desviación estándar del tamaño de los equipos que tienen estudiantes
        """
        team_sizes = [len(team) for team in self.assignments.values() if len(team) > 0]
        return np.std(team_sizes) if team_sizes else 0

    def calculate_average_careers_per_team(self):
        """
        Calcula el promedio de carreras diferentes por equipo
        """
        careers_per_team = []

        for team in self.assignments.values():
            if len(team) >= 2:  # Solo considerar equipos válidos
                unique_careers = len(set(student['Carrera'] for student in team))
                careers_per_team.append(unique_careers)

        return np.mean(careers_per_team) if careers_per_team else 0

    def get_all_metrics(self):
        """
        Calcula y retorna todas las métricas en un diccionario
        """
        metrics = {
            'satisfaccion_promedio': self.calculate_student_satisfaction(),
            'estudiantes_primera_prioridad': self.count_first_priority_assignments(),
            'estudiantes_fuera_preferencias': self.count_outside_preferences(),
            'desafios_sin_equipo': self.count_challenges_without_team(),
            'std_tamaño_equipos': self.calculate_team_size_std(),
            'promedio_carreras_por_equipo': self.calculate_average_careers_per_team()
        }

        return metrics

    def print_metrics(self):
        """
        Imprime todas las métricas de forma legible
        """
        metrics = self.get_all_metrics()

        print("\n=== Métricas de Asignación ===")
        print(f"Satisfacción promedio: {metrics['satisfaccion_promedio']:.3f}")
        print(f"Estudiantes en primera prioridad: {metrics['estudiantes_primera_prioridad']}")
        print(f"Estudiantes fuera de preferencias: {metrics['estudiantes_fuera_preferencias']}")
        print(f"Desafíos sin equipo: {metrics['desafios_sin_equipo']}")
        print(f"Desviación estándar tamaño equipos: {metrics['std_tamaño_equipos']:.3f}")
        print(f"Promedio de carreras por equipo: {metrics['promedio_carreras_por_equipo']:.2f}")

        # Estadísticas adicionales
        total_students = len(self.students)
        print(f"\nPorcentajes:")
        print(f"Primera prioridad: {(metrics['estudiantes_primera_prioridad']/total_students)*100:.1f}%")
        print(f"Fuera de preferencias: {(metrics['estudiantes_fuera_preferencias']/total_students)*100:.1f}%")

In [45]:
# Cargar datos
#data = ... # Tu JSON de entrada

with open('body.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Crear instancia y formar equipos
heuristic = TeamFormationHeuristic(data)
assignments, student_assignments = heuristic.form_teams()

# Calcular métricas
metrics = TeamMetrics(
    assignments=assignments,
    student_assignments=student_assignments,
    students=data['estudiantes'],
    challenges=data['desafios']
)

# Imprimir métricas
metrics.print_metrics()

AttributeError: 'TeamFormationHeuristic' object has no attribute 'count_applications_per_challenge'

In [73]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple

class TeamMetrics:
    def __init__(self, assignments, student_assignments, students, challenges):
        self.assignments = assignments
        self.student_assignments = student_assignments
        self.students = students
        self.challenges = challenges

    def calculate_student_satisfaction(self):
        """
        Calcula la satisfacción promedio de los estudiantes.
        Satisfacción = 1/(índice_preferencia + 1)
        """
        total_satisfaction = 0
        n_students = len(self.students)

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge in student['Postulaciones']:
                preference_index = student['Postulaciones'].index(assigned_challenge)
                satisfaction = 1 / (preference_index + 1)
            else:
                satisfaction = 0
            total_satisfaction += satisfaction

        return total_satisfaction / n_students

    def count_first_priority_assignments(self):
        """
        Cuenta cuántos estudiantes quedaron en su primera preferencia
        """
        first_priority_count = 0

        for student in self.students:
            if (len(student['Postulaciones']) > 0 and
                self.student_assignments.get(student['Nombre']) == student['Postulaciones'][0]):
                first_priority_count += 1

        return first_priority_count

    def count_outside_preferences(self):
        """
        Cuenta cuántos estudiantes quedaron fuera de sus preferencias
        """
        outside_preferences_count = 0

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge not in student['Postulaciones']:
                outside_preferences_count += 1

        return outside_preferences_count

    def count_challenges_without_team(self):
        """
        Cuenta cuántos desafíos quedaron sin equipo (menos de 2 estudiantes)
        """
        challenges_without_team = 0
        challenges_with_students = set(self.assignments.keys())

        # Contar desafíos sin equipo entre los que tienen al menos un estudiante
        for challenge in challenges_with_students:
            if len(self.assignments[challenge]) < 2:
                challenges_without_team += 1

        # Agregar desafíos que no tienen ningún estudiante
        all_challenge_titles = {challenge['Titulo'] for challenge in self.challenges}
        challenges_without_team += len(all_challenge_titles - challenges_with_students)

        return challenges_without_team

    def calculate_team_size_std(self):
        """
        Calcula la desviación estándar del tamaño de los equipos que tienen estudiantes
        """
        team_sizes = [len(team) for team in self.assignments.values() if len(team) > 0]
        return np.std(team_sizes) if team_sizes else 0

    def calculate_average_careers_per_team(self):
        """
        Calcula el promedio de carreras diferentes por equipo
        """
        careers_per_team = []

        for team in self.assignments.values():
            if len(team) >= 2:  # Solo considerar equipos válidos
                unique_careers = len(set(student['Carrera'] for student in team))
                careers_per_team.append(unique_careers)

        return np.mean(careers_per_team) if careers_per_team else 0

    def get_all_metrics(self):
        """
        Calcula y retorna todas las métricas en un diccionario
        """
        metrics = {
            'satisfaccion_promedio': self.calculate_student_satisfaction(),
            'estudiantes_primera_prioridad': self.count_first_priority_assignments(),
            'estudiantes_fuera_preferencias': self.count_outside_preferences(),
            'desafios_sin_equipo': self.count_challenges_without_team(),
            'std_tamaño_equipos': self.calculate_team_size_std(),
            'promedio_carreras_por_equipo': self.calculate_average_careers_per_team()
        }

        return metrics

    def print_metrics(self):
        """
        Imprime todas las métricas de forma legible
        """
        metrics = self.get_all_metrics()

        print("\n=== Métricas de Asignación ===")
        print(f"Satisfacción promedio: {metrics['satisfaccion_promedio']:.3f}")
        print(f"Estudiantes en primera prioridad: {metrics['estudiantes_primera_prioridad']}")
        print(f"Estudiantes fuera de preferencias: {metrics['estudiantes_fuera_preferencias']}")
        print(f"Desafíos sin equipo: {metrics['desafios_sin_equipo']}")
        print(f"Desviación estándar tamaño equipos: {metrics['std_tamaño_equipos']:.3f}")
        print(f"Promedio de carreras por equipo: {metrics['promedio_carreras_por_equipo']:.2f}")

        # Estadísticas adicionales
        total_students = len(self.students)
        print(f"\nPorcentajes:")
        print(f"Primera prioridad: {(metrics['estudiantes_primera_prioridad']/total_students)*100:.1f}%")
        print(f"Fuera de preferencias: {(metrics['estudiantes_fuera_preferencias']/total_students)*100:.1f}%")

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = self.students.copy()
        self.assignment_order = []  # Lista para mantener el orden de asignación
        
    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student in self.unassigned_students:
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1
    
    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True
    
    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student in self.unassigned_students:
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)
    
    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.remove(student)
    
    def get_student_preference_number(self, student: dict, challenge: str) -> str:
        """Obtiene el número de preferencia de un estudiante para un desafío."""
        original_student = next(s for s in self.students if s['Nombre'] == student['Nombre'])
        if challenge in original_student['Postulaciones']:
            return f"Preferencia #{original_student['Postulaciones'].index(challenge) + 1}"
        return "Fuera de preferencias"
    
    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
            if len(team) < 2:  # Saltar equipos incompletos
                continue
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            # Calcular distribución por carrera
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_student_preference_number(student, challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes a desafíos que cumplan las restricciones."""
        for student in self.unassigned_students[:]:
            for challenge in [c['Titulo'] for c in self.challenges]:
                if self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    break
    
    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        for challenge, team in self.assignments.items():
            # Verificar tamaño del equipo
            if not (1<= len(team) < 5):
                print(f"Error: Equipo de tamaño inválido en {challenge}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Verificar que todos los estudiantes estén asignados
        if self.unassigned_students:
            print("Error: Hay estudiantes sin asignar")
            return False
            
        return True
    
    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                self.assign_remaining_students()
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
                        
            self.update_student_priorities(current_challenge)
            
        return self.assignments, self.student_assignments

def main(data):
    heuristic = TeamFormationHeuristic(data)
    assignments, student_assignments = heuristic.form_teams()
    
    if heuristic.validate_assignments():
        print("Asignaciones válidas completadas")
        
        # Crear instancia de TeamMetrics y calcular métricas
        metrics = TeamMetrics(
            assignments=assignments,
            student_assignments=student_assignments,
            students=data['estudiantes'],
            challenges=data['desafios']
        )
        
        # Imprimir métricas
        metrics.print_metrics()
        
        # Imprimir estadísticas detalladas de los equipos
        heuristic.print_team_statistics()
        
    else:
        print("Error en las asignaciones")
    
    return assignments, student_assignments

In [401]:
# Cargar datos
with open('body.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Ejecutar el algoritmo
assignments, student_assignments = main(data)

Error: Equipo con menos de 2 estudiantes en Modelo de control de consumo EPP y ropa de trabajo para la circularidad de elementos en desuso
Error en las asignaciones


In [75]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamMetrics:
    def __init__(self, assignments, student_assignments, students, challenges):
        self.assignments = assignments
        self.student_assignments = student_assignments
        self.students = students
        self.challenges = challenges

    def calculate_student_satisfaction(self):
        """
        Calcula la satisfacción promedio de los estudiantes.
        Satisfacción = 1/(índice_preferencia + 1)
        """
        total_satisfaction = 0
        n_students = len(self.students)

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge in student['Postulaciones']:
                preference_index = student['Postulaciones'].index(assigned_challenge)
                satisfaction = 1 / (preference_index + 1)
            else:
                satisfaction = 0
            total_satisfaction += satisfaction

        return total_satisfaction / n_students

    def count_first_priority_assignments(self):
        """
        Cuenta cuántos estudiantes quedaron en su primera preferencia
        """
        first_priority_count = 0

        for student in self.students:
            if (len(student['Postulaciones']) > 0 and
                self.student_assignments.get(student['Nombre']) == student['Postulaciones'][0]):
                first_priority_count += 1

        return first_priority_count

    def count_outside_preferences(self):
        """
        Cuenta cuántos estudiantes quedaron fuera de sus preferencias
        """
        outside_preferences_count = 0

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge not in student['Postulaciones']:
                outside_preferences_count += 1

        return outside_preferences_count

    def count_challenges_without_team(self):
        """
        Cuenta cuántos desafíos quedaron sin equipo (menos de 2 estudiantes)
        """
        challenges_without_team = 0
        challenges_with_students = set(self.assignments.keys())

        # Contar desafíos sin equipo entre los que tienen al menos un estudiante
        for challenge in challenges_with_students:
            if len(self.assignments[challenge]) < 2:
                challenges_without_team += 1

        # Agregar desafíos que no tienen ningún estudiante
        all_challenge_titles = {challenge['Titulo'] for challenge in self.challenges}
        challenges_without_team += len(all_challenge_titles - challenges_with_students)

        return challenges_without_team

    def calculate_team_size_std(self):
        """
        Calcula la desviación estándar del tamaño de los equipos que tienen estudiantes
        """
        team_sizes = [len(team) for team in self.assignments.values() if len(team) > 0]
        return np.std(team_sizes) if team_sizes else 0

    def calculate_average_careers_per_team(self):
        """
        Calcula el promedio de carreras diferentes por equipo
        """
        careers_per_team = []

        for team in self.assignments.values():
            if len(team) >= 2:  # Solo considerar equipos válidos
                unique_careers = len(set(student['Carrera'] for student in team))
                careers_per_team.append(unique_careers)

        return np.mean(careers_per_team) if careers_per_team else 0

    def get_all_metrics(self):
        """
        Calcula y retorna todas las métricas en un diccionario
        """
        metrics = {
            'satisfaccion_promedio': self.calculate_student_satisfaction(),
            'estudiantes_primera_prioridad': self.count_first_priority_assignments(),
            'estudiantes_fuera_preferencias': self.count_outside_preferences(),
            'desafios_sin_equipo': self.count_challenges_without_team(),
            'std_tamaño_equipos': self.calculate_team_size_std(),
            'promedio_carreras_por_equipo': self.calculate_average_careers_per_team()
        }

        return metrics

    def print_metrics(self):
        """
        Imprime todas las métricas de forma legible
        """
        metrics = self.get_all_metrics()

        print("\n=== Métricas de Asignación ===")
        print(f"Satisfacción promedio: {metrics['satisfaccion_promedio']:.3f}")
        print(f"Estudiantes en primera prioridad: {metrics['estudiantes_primera_prioridad']}")
        print(f"Estudiantes fuera de preferencias: {metrics['estudiantes_fuera_preferencias']}")
        print(f"Desafíos sin equipo: {metrics['desafios_sin_equipo']}")
        print(f"Desviación estándar tamaño equipos: {metrics['std_tamaño_equipos']:.3f}")
        print(f"Promedio de carreras por equipo: {metrics['promedio_carreras_por_equipo']:.2f}")

        # Estadísticas adicionales
        total_students = len(self.students)
        print(f"\nPorcentajes:")
        print(f"Primera prioridad: {(metrics['estudiantes_primera_prioridad']/total_students)*100:.1f}%")
        print(f"Fuera de preferencias: {(metrics['estudiantes_fuera_preferencias']/total_students)*100:.1f}%")

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = copy.deepcopy(self.students)  # Copia para modificar
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }  # Guardar preferencias originales
        self.assignment_order = []  # Lista para mantener el orden de asignación

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos que tienen menos de 2 estudiantes."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 2
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 2:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 2 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            current_team.append(student)
                            self.student_assignments[student['Nombre']] = challenge

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes asegurando equipos de mínimo 2 estudiantes."""
        # Primero, intentar llenar equipos existentes que tengan espacio
        for student in self.unassigned_students[:]:
            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge in self.assignments and len(self.assignments[challenge]) < 4:
                    if self.can_add_student_to_challenge(student, challenge):
                        self.assign_student_to_challenge(student, challenge)
                        break

        # Luego, crear nuevos equipos con los estudiantes restantes
        remaining = self.unassigned_students[:]
        while remaining:
            # Tomar dos estudiantes compatibles para formar un nuevo equipo
            student1 = remaining.pop(0)
            compatible_challenge = None
            compatible_student = None

            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge not in self.assignments:
                    for student2 in remaining:
                        if (self.can_add_student_to_challenge(student1, challenge) and 
                            self.can_add_student_to_challenge(student2, challenge)):
                            compatible_challenge = challenge
                            compatible_student = student2
                            break
                    if compatible_challenge:
                        break

            if compatible_challenge and compatible_student:
                remaining.remove(compatible_student)
                self.assign_student_to_challenge(student1, compatible_challenge)
                self.assign_student_to_challenge(compatible_student, compatible_challenge)
            else:
                # Si no se encuentra un par compatible, asignar a cualquier equipo existente
                assigned = False
                for challenge in self.assignments:
                    if self.can_add_student_to_challenge(student1, challenge):
                        self.assign_student_to_challenge(student1, challenge)
                        assigned = True
                        break
                if not assigned:
                    # Crear nuevo equipo si es necesario
                    new_challenge = next(c['Titulo'] for c in self.challenges 
                                      if c['Titulo'] not in self.assignments)
                    self.assign_student_to_challenge(student1, new_challenge)

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
            if len(team) < 2:  # Saltar equipos incompletos
                continue
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
            
            self.update_student_priorities(current_challenge)
        
        # Asignar estudiantes restantes y completar equipos
        self.assign_remaining_students()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

    # [El resto de los métodos se mantienen igual]

# La clase TeamMetrics se mantiene igual, ya que trabajará con las asignaciones finales
# y las preferencias originales que están en los datos de entrada

def main(data):
    heuristic = TeamFormationHeuristic(data)
    assignments, student_assignments = heuristic.form_teams()
    
    if heuristic.validate_assignments():
        print("Asignaciones válidas completadas")
        
        metrics = TeamMetrics(
            assignments=assignments,
            student_assignments=student_assignments,
            students=data['estudiantes'],  # Usar datos originales para métricas
            challenges=data['desafios']
        )
        
        metrics.print_metrics()
        heuristic.print_team_statistics()
        
    else:
        print("Error en las asignaciones")
    
    return assignments, student_assignments

In [76]:
# Cargar datos
with open('body.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Ejecutar el algoritmo
assignments, student_assignments = main(data)

AttributeError: 'TeamFormationHeuristic' object has no attribute 'count_applications_per_challenge'

Este si

In [403]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamMetrics:
    def __init__(self, assignments, student_assignments, students, challenges):
        self.assignments = assignments
        self.student_assignments = student_assignments
        self.students = students
        self.challenges = challenges

    def calculate_student_satisfaction(self):
        """
        Calcula la satisfacción promedio de los estudiantes.
        Satisfacción = 1/(índice_preferencia + 1)
        """
        total_satisfaction = 0
        n_students = len(self.students)

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge in student['Postulaciones']:
                preference_index = student['Postulaciones'].index(assigned_challenge)
                satisfaction = 1 / (preference_index + 1)
            else:
                satisfaction = 0
            total_satisfaction += satisfaction

        return total_satisfaction / n_students

    def count_first_priority_assignments(self):
        """
        Cuenta cuántos estudiantes quedaron en su primera preferencia
        """
        first_priority_count = 0

        for student in self.students:
            if (len(student['Postulaciones']) > 0 and
                self.student_assignments.get(student['Nombre']) == student['Postulaciones'][0]):
                first_priority_count += 1

        return first_priority_count

    def count_outside_preferences(self):
        """
        Cuenta cuántos estudiantes quedaron fuera de sus preferencias
        """
        outside_preferences_count = 0

        for student in self.students:
            assigned_challenge = self.student_assignments.get(student['Nombre'])
            if assigned_challenge not in student['Postulaciones']:
                outside_preferences_count += 1

        return outside_preferences_count

    def count_challenges_without_team(self):
        """
        Cuenta cuántos desafíos quedaron sin equipo (menos de 2 estudiantes)
        """
        challenges_without_team = 0
        challenges_with_students = set(self.assignments.keys())

        # Contar desafíos sin equipo entre los que tienen al menos un estudiante
        for challenge in challenges_with_students:
            if len(self.assignments[challenge]) < 2:
                challenges_without_team += 1

        # Agregar desafíos que no tienen ningún estudiante
        all_challenge_titles = {challenge['Titulo'] for challenge in self.challenges}
        challenges_without_team += len(all_challenge_titles - challenges_with_students)

        return challenges_without_team

    def calculate_team_size_std(self):
        """
        Calcula la desviación estándar del tamaño de los equipos que tienen estudiantes
        """
        team_sizes = [len(team) for team in self.assignments.values() if len(team) > 0]
        return np.std(team_sizes) if team_sizes else 0

    def calculate_average_careers_per_team(self):
        """
        Calcula el promedio de carreras diferentes por equipo
        """
        careers_per_team = []

        for team in self.assignments.values():
            if len(team) >= 2:  # Solo considerar equipos válidos
                unique_careers = len(set(student['Carrera'] for student in team))
                careers_per_team.append(unique_careers)

        return np.mean(careers_per_team) if careers_per_team else 0

    def get_all_metrics(self):
        """
        Calcula y retorna todas las métricas en un diccionario
        """
        metrics = {
            'satisfaccion_promedio': self.calculate_student_satisfaction(),
            'estudiantes_primera_prioridad': self.count_first_priority_assignments(),
            'estudiantes_fuera_preferencias': self.count_outside_preferences(),
            'desafios_sin_equipo': self.count_challenges_without_team(),
            'std_tamaño_equipos': self.calculate_team_size_std(),
            'promedio_carreras_por_equipo': self.calculate_average_careers_per_team()
        }

        return metrics

    def print_metrics(self):
        """
        Imprime todas las métricas de forma legible
        """
        metrics = self.get_all_metrics()

        print("\n=== Métricas de Asignación ===")
        print(f"Satisfacción promedio: {metrics['satisfaccion_promedio']:.3f}")
        print(f"Estudiantes en primera prioridad: {metrics['estudiantes_primera_prioridad']}")
        print(f"Estudiantes fuera de preferencias: {metrics['estudiantes_fuera_preferencias']}")
        print(f"Desafíos sin equipo: {metrics['desafios_sin_equipo']}")
        print(f"Desviación estándar tamaño equipos: {metrics['std_tamaño_equipos']:.3f}")
        print(f"Promedio de carreras por equipo: {metrics['promedio_carreras_por_equipo']:.2f}")

        # Estadísticas adicionales
        total_students = len(self.students)
        print(f"\nPorcentajes:")
        print(f"Primera prioridad: {(metrics['estudiantes_primera_prioridad']/total_students)*100:.1f}%")
        print(f"Fuera de preferencias: {(metrics['estudiantes_fuera_preferencias']/total_students)*100:.1f}%")

In [404]:
class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = copy.deepcopy(self.students)
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student in self.unassigned_students:
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student in self.unassigned_students:
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        if student in self.unassigned_students:
            self.unassigned_students.remove(student)

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos que tienen menos de 2 estudiantes."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 2
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 2:
                    for student in other_team[:]:
                        if len(current_team) < 2 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            current_team.append(student)
                            self.student_assignments[student['Nombre']] = challenge

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes asegurando equipos de mínimo 2 estudiantes."""
        # Primero, intentar llenar equipos existentes que tengan espacio
        for student in self.unassigned_students[:]:
            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge in self.assignments and len(self.assignments[challenge]) < 4:
                    if self.can_add_student_to_challenge(student, challenge):
                        self.assign_student_to_challenge(student, challenge)
                        break

        # Crear nuevos equipos con los estudiantes restantes
        remaining = self.unassigned_students[:]
        while remaining:
            student1 = remaining.pop(0)
            compatible_challenge = None
            compatible_student = None

            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge not in self.assignments:
                    for student2 in remaining:
                        if (self.can_add_student_to_challenge(student1, challenge) and 
                            self.can_add_student_to_challenge(student2, challenge)):
                            compatible_challenge = challenge
                            compatible_student = student2
                            break
                    if compatible_challenge:
                        break

            if compatible_challenge and compatible_student:
                remaining.remove(compatible_student)
                self.assign_student_to_challenge(student1, compatible_challenge)
                self.assign_student_to_challenge(compatible_student, compatible_challenge)
            else:
                # Si no se encuentra un par compatible, asignar a cualquier equipo existente
                assigned = False
                for challenge in self.assignments:
                    if self.can_add_student_to_challenge(student1, challenge):
                        self.assign_student_to_challenge(student1, challenge)
                        assigned = True
                        break
                if not assigned:
                    # Crear nuevo equipo si es necesario
                    new_challenge = next(c['Titulo'] for c in self.challenges 
                                      if c['Titulo'] not in self.assignments)
                    self.assign_student_to_challenge(student1, new_challenge)

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
            if len(team) < 2:
                continue
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = sum(len(team) for team in self.assignments.values())
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            if len(team) < 2:
                print(f"Error: Equipo con menos de {len(team)} estudiantes en {challenge}")
                return False
            if len(team) > 4:
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}")
                return False
                
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        return True

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
            
            self.update_student_priorities(current_challenge)
        
        self.assign_remaining_students()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

def main(data):
    heuristic = TeamFormationHeuristic(data)
    assignments, student_assignments = heuristic.form_teams()
    
    if heuristic.validate_assignments():
        print("Asignaciones válidas completadas")
        
        metrics = TeamMetrics(
            assignments=assignments,
            student_assignments=student_assignments,
            students=data['estudiantes'],  # Usar datos originales para métricas
            challenges=data['desafios']
        )
        
        metrics.print_metrics()
        heuristic.print_team_statistics()
        
    else:
        print("Error en las asignaciones")
    
    return assignments, student_assignments

In [414]:
class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = copy.deepcopy(self.students)
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student in self.unassigned_students:
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student in self.unassigned_students:
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        if student in self.unassigned_students:
            self.unassigned_students.remove(student)

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            current_team.append(student)
                            self.student_assignments[student['Nombre']] = challenge

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes priorizando equipos de al menos 3 estudiantes."""
        # Primero, completar equipos existentes hasta 3 integrantes
        for challenge in list(self.assignments.keys()):
            if len(self.assignments[challenge]) < 3:
                for student in self.unassigned_students[:]:
                    if self.can_add_student_to_challenge(student, challenge):
                        self.assign_student_to_challenge(student, challenge)
                        if len(self.assignments[challenge]) >= 3:
                            break

        # Luego, crear nuevos equipos con los estudiantes restantes
        remaining = self.unassigned_students[:]
        while len(remaining) >= 3:  # Asegurar que podemos formar equipos de al menos 3
            student1 = remaining.pop(0)
            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge not in self.assignments:
                    compatible_students = []
                    for student in remaining:
                        if self.can_add_student_to_challenge(student, challenge):
                            compatible_students.append(student)
                            if len(compatible_students) >= 2:  # Necesitamos 2 más para formar equipo de 3
                                break
                    
                    if len(compatible_students) >= 2:
                        self.assign_student_to_challenge(student1, challenge)
                        for compatible_student in compatible_students[:2]:
                            remaining.remove(compatible_student)
                            self.assign_student_to_challenge(compatible_student, challenge)
                        break

        # Si quedan estudiantes, intentar agregarlos a equipos existentes
        for student in remaining[:]:
            added = False
            for challenge in self.assignments:
                if len(self.assignments[challenge]) < 4 and self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    added = True
                    break
            
            if not added and remaining:
                # Si no se pudo agregar a equipos existentes, crear nuevo equipo
                available_challenges = [c['Titulo'] for c in self.challenges if c['Titulo'] not in self.assignments]
                if available_challenges:
                    new_challenge = available_challenges[0]
                    self.assign_student_to_challenge(student, new_challenge)

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = sum(len(team) for team in self.assignments.values())
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Cambio en la validación del tamaño mínimo
            if len(team) < 2:
                print(f"Error: Equipo con menos de {len(team)} estudiantes en {challenge}")
                return False
            if len(team) > 4:
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}")
                return False
                
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        return True

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
            
            self.update_student_priorities(current_challenge)
        
        self.assign_remaining_students()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [416]:
class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = copy.deepcopy(self.students)
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student in self.unassigned_students:
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student in self.unassigned_students:
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        if student in self.unassigned_students:
            self.unassigned_students.remove(student)

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            current_team.append(student)
                            self.student_assignments[student['Nombre']] = challenge

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes priorizando equipos de al menos 3 estudiantes."""
        # Primero, completar equipos existentes hasta 3 integrantes
        for challenge in list(self.assignments.keys()):
            if len(self.assignments[challenge]) < 3:
                for student in self.unassigned_students[:]:
                    if self.can_add_student_to_challenge(student, challenge):
                        self.assign_student_to_challenge(student, challenge)
                        if len(self.assignments[challenge]) >= 3:
                            break

        # Luego, crear nuevos equipos con los estudiantes restantes
        remaining = self.unassigned_students[:]
        while len(remaining) >= 3:  # Asegurar que podemos formar equipos de al menos 3
            student1 = remaining.pop(0)
            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge not in self.assignments:
                    compatible_students = []
                    for student in remaining:
                        if self.can_add_student_to_challenge(student, challenge):
                            compatible_students.append(student)
                            if len(compatible_students) >= 2:  # Necesitamos 2 más para formar equipo de 3
                                break
                    
                    if len(compatible_students) >= 2:
                        self.assign_student_to_challenge(student1, challenge)
                        for compatible_student in compatible_students[:2]:
                            remaining.remove(compatible_student)
                            self.assign_student_to_challenge(compatible_student, challenge)
                        break

        # Si quedan estudiantes, intentar agregarlos a equipos existentes
        for student in remaining[:]:
            added = False
            for challenge in self.assignments:
                if len(self.assignments[challenge]) < 4 and self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    added = True
                    break
            
            if not added and remaining:
                # Si no se pudo agregar a equipos existentes, crear nuevo equipo
                available_challenges = [c['Titulo'] for c in self.challenges if c['Titulo'] not in self.assignments]
                if available_challenges:
                    new_challenge = available_challenges[0]
                    self.assign_student_to_challenge(student, new_challenge)

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = sum(len(team) for team in self.assignments.values())
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            random.shuffle(priority_students)
            for student in priority_students:
                if self.can_add_student_to_challenge(student, current_challenge):
                    self.assign_student_to_challenge(student, current_challenge)
                    if len(self.assignments[current_challenge]) >= 4:
                        break
            
            self.update_student_priorities(current_challenge)
        
        self.assign_remaining_students()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [418]:
class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = copy.deepcopy(self.students)
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student in self.unassigned_students:
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student in self.unassigned_students:
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str):
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        if student in self.unassigned_students:
            self.unassigned_students.remove(student)

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            current_team.append(student)
                            self.student_assignments[student['Nombre']] = challenge

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes priorizando equipos de al menos 3 estudiantes."""
        # Primero, completar equipos existentes hasta 3 integrantes
        for challenge in list(self.assignments.keys()):
            if len(self.assignments[challenge]) < 3:
                for student in self.unassigned_students[:]:
                    if self.can_add_student_to_challenge(student, challenge):
                        self.assign_student_to_challenge(student, challenge)
                        if len(self.assignments[challenge]) >= 3:
                            break

        # Luego, crear nuevos equipos con los estudiantes restantes
        remaining = self.unassigned_students[:]
        while len(remaining) >= 2:  # Cambiado de 3 a 2 para garantizar mínimo 2
            student1 = remaining.pop(0)
            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge not in self.assignments:
                    compatible_student = None
                    # Buscar al menos un estudiante compatible para formar equipo de 2
                    for student2 in remaining:
                        if self.can_add_student_to_challenge(student2, challenge):
                            compatible_student = student2
                            # Si encontramos un segundo estudiante compatible, formamos el equipo
                            self.assign_student_to_challenge(student1, challenge)
                            remaining.remove(compatible_student)
                            self.assign_student_to_challenge(compatible_student, challenge)
                            
                            # Intentar agregar un tercer estudiante si es posible
                            for student3 in remaining[:]:
                                if self.can_add_student_to_challenge(student3, challenge):
                                    self.assign_student_to_challenge(student3, challenge)
                                    break
                            break
                    if compatible_student:
                        break
            
            # Si no pudimos asignar al estudiante1, intentar agregarlo a equipos existentes
            if student1 in remaining:
                for challenge in self.assignments:
                    if self.can_add_student_to_challenge(student1, challenge):
                        self.assign_student_to_challenge(student1, challenge)
                        break

        # Si quedan estudiantes individuales, intentar agregarlos a equipos existentes
        for student in remaining[:]:
            assigned = False
            for challenge in self.assignments:
                if self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    assigned = True
                    break
            
            if not assigned:
                # Buscar otro estudiante sin asignar para formar un nuevo equipo
                other_unassigned = [s for s in remaining if s != student]
                if other_unassigned:
                    student2 = other_unassigned[0]
                    remaining.remove(student2)
                    # Encontrar un desafío disponible para el nuevo equipo
                    available_challenges = [c['Titulo'] for c in self.challenges if c['Titulo'] not in self.assignments]
                    if available_challenges:
                        new_challenge = available_challenges[0]
                        self.assign_student_to_challenge(student, new_challenge)
                        self.assign_student_to_challenge(student2, new_challenge)

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = sum(len(team) for team in self.assignments.values())
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            priority_students = [
                student for student in self.unassigned_students 
                if student['Postulaciones'] and student['Postulaciones'][0] == current_challenge
            ]
            
            # Asegurarse de formar al menos equipos de 2 estudiantes
            if len(priority_students) >= 2:
                random.shuffle(priority_students)
                # Asignar primero dos estudiantes
                for i in range(2):
                    student = priority_students[i]
                    if self.can_add_student_to_challenge(student, current_challenge):
                        self.assign_student_to_challenge(student, current_challenge)
                
                # Luego intentar agregar más hasta 4
                for student in priority_students[2:]:
                    if self.can_add_student_to_challenge(student, current_challenge):
                        self.assign_student_to_challenge(student, current_challenge)
                        if len(self.assignments[current_challenge]) >= 4:
                            break
            
            self.update_student_priorities(current_challenge)
        
        self.assign_remaining_students()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [422]:
class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = set(student['Nombre'] for student in self.students)
        self.students_dict = {student['Nombre']: student for student in self.students}
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones por desafío."""
        challenge_counts = defaultdict(int)
        for student_name in self.unassigned_students:
            student = self.students_dict[student_name]
            for challenge in student['Postulaciones']:
                challenge_counts[challenge] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        # Verificar si el estudiante ya está asignado
        if student['Nombre'] in self.student_assignments:
            return False
            
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if student['Nombre'] in self.student_assignments:
            return False

        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
            
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.discard(student['Nombre'])
        return True

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            # Remover estudiante del equipo anterior
                            other_team.remove(student)
                            del self.student_assignments[student['Nombre']]
                            
                            # Asignar al nuevo equipo
                            self.assign_student_to_challenge(student, challenge)
                            if len(current_team) >= 3:
                                break

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes priorizando equipos de al menos 3 estudiantes."""
        # Primero, completar equipos existentes hasta 3 integrantes
        for challenge in list(self.assignments.keys()):
            if len(self.assignments[challenge]) < 3:
                unassigned = list(self.unassigned_students)
                for student_name in unassigned:
                    student = self.students_dict[student_name]
                    if self.can_add_student_to_challenge(student, challenge):
                        self.assign_student_to_challenge(student, challenge)
                        if len(self.assignments[challenge]) >= 3:
                            break

        # Luego, crear nuevos equipos con los estudiantes restantes
        while len(self.unassigned_students) >= 2:
            unassigned = list(self.unassigned_students)
            if not unassigned:
                break
                
            student1 = self.students_dict[unassigned[0]]
            found_team = False
            
            # Buscar un desafío disponible y estudiantes compatibles
            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge not in self.assignments:
                    compatible_students = []
                    for student_name in list(self.unassigned_students)[1:]:
                        student2 = self.students_dict[student_name]
                        if self.can_add_student_to_challenge(student2, challenge):
                            compatible_students.append(student2)
                            if len(compatible_students) >= 1:
                                # Formar equipo con al menos 2 estudiantes
                                self.assign_student_to_challenge(student1, challenge)
                                self.assign_student_to_challenge(compatible_students[0], challenge)
                                found_team = True
                                break
                    if found_team:
                        break
            
            if not found_team and student1['Nombre'] in self.unassigned_students:
                # Si no se pudo formar equipo nuevo, intentar agregar a equipos existentes
                for challenge in self.assignments:
                    if self.can_add_student_to_challenge(student1, challenge):
                        self.assign_student_to_challenge(student1, challenge)
                        break

        # Intentar asignar estudiantes restantes a equipos existentes
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            for challenge in self.assignments:
                if self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    break

    def verify_integrity(self) -> bool:
        """Verifica la integridad de las asignaciones."""
        assigned_students = set(self.student_assignments.keys())
        team_students = set()
        for team in self.assignments.values():
            for student in team:
                team_students.add(student['Nombre'])
                
        all_students = set(student['Nombre'] for student in self.students)
        
        if len(assigned_students) != len(team_students):
            print(f"Discrepancia entre estudiantes asignados ({len(assigned_students)}) y estudiantes en equipos ({len(team_students)})")
            return False
            
        if len(assigned_students) > len(all_students):
            print(f"Hay más estudiantes asignados ({len(assigned_students)}) que estudiantes totales ({len(all_students)})")
            return False
            
        return True

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        if not self.verify_integrity():
            return False
            
        total_assigned_students = len(self.student_assignments)
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True
    
    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = 3 if career == "Ingeniería Civil Telemática" else 1
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            priority_students = [
                self.students_dict[student_name] for student_name in self.unassigned_students 
                if current_challenge in self.students_dict[student_name]['Postulaciones']
            ]
            
            # Asegurarse de formar al menos equipos de 2 estudiantes
            if len(priority_students) >= 2:
                random.shuffle(priority_students)
                # Asignar primero dos estudiantes
                for i in range(2):
                    student = priority_students[i]
                    if self.can_add_student_to_challenge(student, current_challenge):
                        self.assign_student_to_challenge(student, current_challenge)
                
                # Luego intentar agregar más hasta 4
                for student in priority_students[2:]:
                    if self.can_add_student_to_challenge(student, current_challenge):
                        self.assign_student_to_challenge(student, current_challenge)
                        if len(self.assignments[current_challenge]) >= 4:
                            break
            
            self.update_student_priorities(current_challenge)
        
        self.assign_remaining_students()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [426]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = set(student['Nombre'] for student in self.students)
        self.students_dict = {student['Nombre']: student for student in self.students}
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones de prioridad 1 por desafío."""
        challenge_counts = defaultdict(int)
        for student_name in self.unassigned_students:
            student = self.students_dict[student_name]
            if student['Postulaciones']:  # Si tiene postulaciones pendientes
                # Solo contar la primera prioridad actual
                challenge_counts[student['Postulaciones'][0]] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        # Verificar si el estudiante ya está asignado
        if student['Nombre'] in self.student_assignments:
            return False
            
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if student['Nombre'] in self.student_assignments:
            return False

        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
            
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.discard(student['Nombre'])
        return True

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes cuando sea posible."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            del self.student_assignments[student['Nombre']]
                            self.assign_student_to_challenge(student, challenge)
                            if len(current_team) >= 3:
                                break

    def assign_remaining_students(self):
        """Asigna los estudiantes restantes priorizando equipos de al menos 2 estudiantes."""
        # Primero, completar equipos existentes hasta 3 integrantes
        for challenge in list(self.assignments.keys()):
            if len(self.assignments[challenge]) < 3:
                unassigned = list(self.unassigned_students)
                for student_name in unassigned:
                    student = self.students_dict[student_name]
                    if self.can_add_student_to_challenge(student, challenge):
                        self.assign_student_to_challenge(student, challenge)
                        if len(self.assignments[challenge]) >= 3:
                            break

        # Luego, crear nuevos equipos con los estudiantes restantes
        while len(self.unassigned_students) >= 2:
            unassigned = list(self.unassigned_students)
            if not unassigned:
                break
                
            student1 = self.students_dict[unassigned[0]]
            found_team = False
            
            # Buscar un desafío disponible y estudiantes compatibles
            for challenge in [c['Titulo'] for c in self.challenges]:
                if challenge not in self.assignments:
                    compatible_students = []
                    for student_name in list(self.unassigned_students)[1:]:
                        student2 = self.students_dict[student_name]
                        if self.can_add_student_to_challenge(student2, challenge):
                            compatible_students.append(student2)
                            if len(compatible_students) >= 1:
                                # Formar equipo con al menos 2 estudiantes
                                self.assign_student_to_challenge(student1, challenge)
                                self.assign_student_to_challenge(compatible_students[0], challenge)
                                found_team = True
                                break
                    if found_team:
                        break
            
            if not found_team and student1['Nombre'] in self.unassigned_students:
                # Si no se pudo formar equipo nuevo, intentar agregar a equipos existentes
                for challenge in self.assignments:
                    if self.can_add_student_to_challenge(student1, challenge):
                        self.assign_student_to_challenge(student1, challenge)
                        break

        # Intentar asignar estudiantes restantes a equipos existentes
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            for challenge in self.assignments:
                if self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    break

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = self.get_career_limit(career)
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = len(self.student_assignments)
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}, {len(team)}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            # Obtener el desafío con más postulaciones de prioridad 1
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            # Obtener estudiantes que tienen este desafío como primera prioridad actual
            priority_students = [
                self.students_dict[student_name] for student_name in self.unassigned_students
                if student_name in self.students_dict and
                self.students_dict[student_name]['Postulaciones'] and
                self.students_dict[student_name]['Postulaciones'][0] == current_challenge
            ]
            
            if priority_students:
                # Asignación aleatoria entre estudiantes con primera prioridad
                random.shuffle(priority_students)
                
                # Asignar estudiantes respetando restricciones
                for student in priority_students:
                    if self.can_add_student_to_challenge(student, current_challenge):
                        self.assign_student_to_challenge(student, current_challenge)
                        if len(self.assignments[current_challenge]) >= 4:
                            break
            
            # Actualizar prioridades de los no asignados
            self.update_student_priorities(current_challenge)
        
        # Manejar estudiantes restantes y completar equipos
        self.assign_remaining_students()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [430]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = set(student['Nombre'] for student in self.students)
        self.students_dict = {student['Nombre']: student for student in self.students}
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones de prioridad 1 por desafío."""
        challenge_counts = defaultdict(int)
        for student_name in self.unassigned_students:
            student = self.students_dict[student_name]
            if student['Postulaciones']:  # Si tiene postulaciones pendientes
                # Solo contar la primera prioridad actual
                challenge_counts[student['Postulaciones'][0]] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        # Verificar si el estudiante ya está asignado
        if student['Nombre'] in self.student_assignments:
            return False
            
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if student['Nombre'] in self.student_assignments:
            return False

        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
            
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.discard(student['Nombre'])
        return True

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes cuando sea posible."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            del self.student_assignments[student['Nombre']]
                            self.assign_student_to_challenge(student, challenge)
                            if len(current_team) >= 3:
                                break

    def get_compatible_students_for_challenge(self, challenge: str, students: List[dict]) -> List[dict]:
        """
        Obtiene una lista de estudiantes compatibles para un desafío,
        considerando las restricciones de carrera.
        """
        compatible_students = []
        temp_team = []  # Para simular el equipo mientras verificamos compatibilidad
        
        for student in students:
            # Crear una copia temporal del equipo para verificar
            temp_team_copy = temp_team.copy()
            temp_team_copy.append(student)
            
            # Verificar restricciones de carrera
            career_counts = defaultdict(int)
            for temp_student in temp_team_copy:
                career_counts[temp_student['Carrera']] += 1
                if career_counts[temp_student['Carrera']] > self.get_career_limit(temp_student['Carrera']):
                    break
            else:
                # Si llegamos aquí, el estudiante es compatible
                compatible_students.append(student)
                temp_team.append(student)
                
                # Si ya tenemos 4 estudiantes compatibles, es suficiente
                if len(compatible_students) >= 4:
                    break
        
        return compatible_students

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = self.get_career_limit(career)
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = len(self.student_assignments)
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}, {len(team)}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}, {len(team)}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def assign_remaining_students_in_pairs(self):
        """
        Asigna los estudiantes restantes asegurando que siempre se formen equipos
        de al menos 2 estudiantes.
        """
        while len(self.unassigned_students) >= 2:
            # Tomar todos los estudiantes no asignados
            remaining_students = [self.students_dict[name] for name in self.unassigned_students]
            found_pair = False
            
            # Intentar primero completar equipos existentes que tengan espacio
            for challenge, team in self.assignments.items():
                if len(team) < 4:  # Solo si hay espacio en el equipo
                    compatible_for_team = []
                    for student in remaining_students:
                        if self.can_add_student_to_challenge(student, challenge):
                            compatible_for_team.append(student)
                            if len(compatible_for_team) >= 2:
                                # Asignar dos estudiantes compatibles al equipo existente
                                for compatible_student in compatible_for_team[:2]:
                                    self.assign_student_to_challenge(compatible_student, challenge)
                                found_pair = True
                                break
                    if found_pair:
                        break
            
            if not found_pair:
                # Si no pudimos agregar a equipos existentes, intentar crear un nuevo equipo
                for challenge in [c['Titulo'] for c in self.challenges]:
                    if challenge not in self.assignments:
                        compatible_students = self.get_compatible_students_for_challenge(challenge, remaining_students)
                        if len(compatible_students) >= 2:
                            # Crear nuevo equipo con al menos 2 estudiantes
                            for i in range(2):
                                self.assign_student_to_challenge(compatible_students[i], challenge)
                            found_pair = True
                            break
            
            # Si no pudimos asignar ningún par, salimos del ciclo
            if not found_pair:
                break

        # Si quedan estudiantes individuales, intentar agregarlos a equipos existentes
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            for challenge, team in self.assignments.items():
                if len(team) >= 2 and self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    break

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            # Obtener el desafío con más postulaciones de prioridad 1
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            # Obtener estudiantes que tienen este desafío como primera prioridad actual
            priority_students = [
                self.students_dict[student_name] for student_name in self.unassigned_students
                if student_name in self.students_dict and
                self.students_dict[student_name]['Postulaciones'] and
                self.students_dict[student_name]['Postulaciones'][0] == current_challenge
            ]
            
            if len(priority_students) >= 2:
                # Obtener estudiantes compatibles considerando restricciones
                random.shuffle(priority_students)  # Aleatorizar antes de verificar compatibilidad
                compatible_students = self.get_compatible_students_for_challenge(current_challenge, priority_students)
                
                # Solo formar equipo si tenemos al menos 2 estudiantes compatibles
                if len(compatible_students) >= 2:
                    # Asignar primero los dos estudiantes mínimos requeridos
                    for i in range(2):
                        self.assign_student_to_challenge(compatible_students[i], current_challenge)
                    
                    # Luego intentar agregar más hasta 4
                    for student in compatible_students[2:]:
                        if len(self.assignments[current_challenge]) < 4:
                            self.assign_student_to_challenge(student, current_challenge)
                        else:
                            break
            
            # Actualizar prioridades de los no asignados
            self.update_student_priorities(current_challenge)
        
        # Ahora manejaremos los estudiantes restantes de manera diferente
        self.assign_remaining_students_in_pairs()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [ ]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = set(student['Nombre'] for student in self.students)
        self.students_dict = {student['Nombre']: student for student in self.students}
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones de prioridad 1 por desafío."""
        challenge_counts = defaultdict(int)
        for student_name in self.unassigned_students:
            student = self.students_dict[student_name]
            if student['Postulaciones']:  # Si tiene postulaciones pendientes
                # Solo contar la primera prioridad actual
                challenge_counts[student['Postulaciones'][0]] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        # Verificar si el estudiante ya está asignado
        if student['Nombre'] in self.student_assignments:
            return False
            
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if student['Nombre'] in self.student_assignments:
            return False

        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
            
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.discard(student['Nombre'])
        return True

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes cuando sea posible."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            del self.student_assignments[student['Nombre']]
                            self.assign_student_to_challenge(student, challenge)
                            if len(current_team) >= 3:
                                break

    def get_compatible_students_for_challenge(self, challenge: str, students: List[dict]) -> List[dict]:
        """
        Obtiene una lista de estudiantes compatibles para un desafío,
        considerando las restricciones de carrera.
        """
        compatible_students = []
        temp_team = []  # Para simular el equipo mientras verificamos compatibilidad
        
        for student in students:
            # Crear una copia temporal del equipo para verificar
            temp_team_copy = temp_team.copy()
            temp_team_copy.append(student)
            
            # Verificar restricciones de carrera
            career_counts = defaultdict(int)
            for temp_student in temp_team_copy:
                career_counts[temp_student['Carrera']] += 1
                if career_counts[temp_student['Carrera']] > self.get_career_limit(temp_student['Carrera']):
                    break
            else:
                # Si llegamos aquí, el estudiante es compatible
                compatible_students.append(student)
                temp_team.append(student)
                
                # Si ya tenemos 4 estudiantes compatibles, es suficiente
                if len(compatible_students) >= 4:
                    break
        
        return compatible_students

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = self.get_career_limit(career)
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = len(self.student_assignments)
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}, {len(team)}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}, {len(team)}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def assign_remaining_students_in_pairs(self):
        """
        Asigna los estudiantes restantes asegurando que siempre se formen equipos
        de al menos 2 estudiantes.
        """
        while len(self.unassigned_students) >= 2:
            # Tomar todos los estudiantes no asignados
            remaining_students = [self.students_dict[name] for name in self.unassigned_students]
            found_pair = False
            
            # Intentar primero completar equipos existentes que tengan espacio
            for challenge, team in self.assignments.items():
                if len(team) < 3:  # Solo si hay espacio en el equipo
                    compatible_for_team = []
                    for student in remaining_students:
                        if self.can_add_student_to_challenge(student, challenge):
                            compatible_for_team.append(student)
                            if len(compatible_for_team) >= 2:
                                # Asignar dos estudiantes compatibles al equipo existente
                                for compatible_student in compatible_for_team[:2]:
                                    self.assign_student_to_challenge(compatible_student, challenge)
                                found_pair = True
                                break
                    if found_pair:
                        break
            
            if not found_pair:
                # Si no pudimos agregar a equipos existentes, intentar crear un nuevo equipo
                for challenge in [c['Titulo'] for c in self.challenges]:
                    if challenge not in self.assignments:
                        compatible_students = self.get_compatible_students_for_challenge(challenge, remaining_students)
                        if len(compatible_students) >= 2:
                            # Crear nuevo equipo con al menos 2 estudiantes
                            for i in range(2):
                                self.assign_student_to_challenge(compatible_students[i], challenge)
                            found_pair = True
                            break
            
            # Si no pudimos asignar ningún par, salimos del ciclo
            if not found_pair:
                break

        # Si quedan estudiantes individuales, intentar agregarlos a equipos existentes
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            for challenge, team in self.assignments.items():
                if len(team) >= 2 and self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    break

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            # Obtener el desafío con más postulaciones de prioridad 1
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            # Obtener estudiantes que tienen este desafío como primera prioridad actual
            priority_students = [
                self.students_dict[student_name] for student_name in self.unassigned_students
                if student_name in self.students_dict and
                self.students_dict[student_name]['Postulaciones'] and
                self.students_dict[student_name]['Postulaciones'][0] == current_challenge
            ]
            
            if len(priority_students) >= 2:
                # Obtener estudiantes compatibles considerando restricciones
                random.shuffle(priority_students)  # Aleatorizar antes de verificar compatibilidad
                compatible_students = self.get_compatible_students_for_challenge(current_challenge, priority_students)
                
                # Solo formar equipo si tenemos al menos 2 estudiantes compatibles
                if len(compatible_students) >= 2:
                    # Asignar primero los dos estudiantes mínimos requeridos
                    for i in range(2):
                        self.assign_student_to_challenge(compatible_students[i], current_challenge)
                    
                    # Luego intentar agregar más hasta 4
                    for student in compatible_students[2:]:
                        if len(self.assignments[current_challenge]) < 4:
                            self.assign_student_to_challenge(student, current_challenge)
                        else:
                            break
            
            # Actualizar prioridades de los no asignados
            self.update_student_priorities(current_challenge)
        
        # Ahora manejaremos los estudiantes restantes de manera diferente
        self.assign_remaining_students_in_pairs()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [ ]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = set(student['Nombre'] for student in self.students)
        self.students_dict = {student['Nombre']: student for student in self.students}
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones de prioridad 1 por desafío."""
        challenge_counts = defaultdict(int)
        for student_name in self.unassigned_students:
            student = self.students_dict[student_name]
            if student['Postulaciones']:  # Si tiene postulaciones pendientes
                # Solo contar la primera prioridad actual
                challenge_counts[student['Postulaciones'][0]] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        # Verificar si el estudiante ya está asignado
        if student['Nombre'] in self.student_assignments:
            return False
            
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo - Estricto máximo de 4
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        career_limit = self.get_career_limit(student['Carrera'])
        if career_counts[student['Carrera']] >= career_limit:
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if student['Nombre'] in self.student_assignments:
            return False

        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
            
        if len(self.assignments[challenge]) >= 4:
            return False
            
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.discard(student['Nombre'])
        return True

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def get_compatible_students_for_challenge(self, challenge: str, students: List[dict]) -> List[dict]:
        """
        Obtiene una lista de estudiantes compatibles para un desafío,
        considerando las restricciones de carrera.
        """
        compatible_students = []
        temp_team = []  # Para simular el equipo mientras verificamos compatibilidad
        
        for student in students:
            # Crear una copia temporal del equipo para verificar
            temp_team_copy = temp_team.copy()
            temp_team_copy.append(student)
            
            # Verificar restricciones de carrera
            career_counts = defaultdict(int)
            for temp_student in temp_team_copy:
                career_counts[temp_student['Carrera']] += 1
                if career_counts[temp_student['Carrera']] > self.get_career_limit(temp_student['Carrera']):
                    break
            else:
                # Si llegamos aquí, el estudiante es compatible
                compatible_students.append(student)
                temp_team.append(student)
                
                # Si ya tenemos 4 estudiantes compatibles, es suficiente
                if len(compatible_students) >= 4:
                    break
        
        return compatible_students

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = self.get_career_limit(career)
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = len(self.student_assignments)
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}, {len(team)}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}, {len(team)}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def assign_remaining_students_in_pairs(self):
        """
        Asigna los estudiantes restantes asegurando que siempre se formen equipos
        de al menos 2 estudiantes y máximo 4.
        """
        while len(self.unassigned_students) >= 2:
            # Tomar todos los estudiantes no asignados
            remaining_students = [self.students_dict[name] for name in self.unassigned_students]
            found_pair = False
            
            # Intentar primero completar equipos existentes que tengan espacio
            for challenge, team in self.assignments.items():
                if len(team) < 3:  # Priorizar equipos pequeños primero
                    compatible_for_team = []
                    for student in remaining_students:
                        if self.can_add_student_to_challenge(student, challenge):
                            compatible_for_team.append(student)
                            if len(compatible_for_team) >= 2:
                                # Asignar dos estudiantes compatibles al equipo existente
                                for compatible_student in compatible_for_team[:2]:
                                    self.assign_student_to_challenge(compatible_student, challenge)
                                found_pair = True
                                break
                    if found_pair:
                        break
            
            if not found_pair:
                # Si no pudimos agregar a equipos pequeños, intentar con equipos que tienen espacio
                for challenge, team in self.assignments.items():
                    if len(team) < 4:  # Solo si hay espacio en el equipo
                        compatible_for_team = []
                        for student in remaining_students:
                            if self.can_add_student_to_challenge(student, challenge):
                                compatible_for_team.append(student)
                                # Solo necesitamos un estudiante compatible para equipos existentes
                                if len(compatible_for_team) >= 1:
                                    self.assign_student_to_challenge(compatible_for_team[0], challenge)
                                    found_pair = True
                                    break
                        if found_pair:
                            break
            
            if not found_pair:
                # Si no pudimos agregar a equipos existentes, intentar crear un nuevo equipo
                for challenge in [c['Titulo'] for c in self.challenges]:
                    if challenge not in self.assignments:
                        compatible_students = self.get_compatible_students_for_challenge(challenge, remaining_students)
                        if len(compatible_students) >= 2:
                            # Crear nuevo equipo con al menos 2 estudiantes, máximo 4
                            max_to_assign = min(4, len(compatible_students))
                            for i in range(max_to_assign):
                                self.assign_student_to_challenge(compatible_students[i], challenge)
                            found_pair = True
                            break
            
            # Si no pudimos asignar ningún par, salimos del ciclo
            if not found_pair:
                break

        # Si quedan estudiantes individuales, intentar agregarlos a equipos existentes
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            for challenge, team in self.assignments.items():
                if len(team) >= 2 and len(team) < 4 and self.can_add_student_to_challenge(student, challenge):
                    self.assign_student_to_challenge(student, challenge)
                    break

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes cuando sea posible y respetando máximo 4."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge or len(current_team) >= 4:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            del self.student_assignments[student['Nombre']]
                            self.assign_student_to_challenge(student, challenge)
                            if len(current_team) >= 3:
                                break

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            # Obtener el desafío con más postulaciones de prioridad 1
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            # Obtener estudiantes que tienen este desafío como primera prioridad actual
            priority_students = [
                self.students_dict[student_name] for student_name in self.unassigned_students
                if student_name in self.students_dict and
                self.students_dict[student_name]['Postulaciones'] and
                self.students_dict[student_name]['Postulaciones'][0] == current_challenge
            ]
            
            if len(priority_students) >= 2:
                # Obtener estudiantes compatibles considerando restricciones
                random.shuffle(priority_students)  # Aleatorizar antes de verificar compatibilidad
                compatible_students = self.get_compatible_students_for_challenge(current_challenge, priority_students)
                
                # Solo formar equipo si tenemos al menos 2 estudiantes compatibles
                if len(compatible_students) >= 2:
                    # Asignar primero los dos estudiantes mínimos requeridos
                    for i in range(2):
                        self.assign_student_to_challenge(compatible_students[i], current_challenge)
                    
                    # Luego intentar agregar más hasta 4 (estricto)
                    remaining = min(len(compatible_students[2:]), 2)  # Máximo 2 más
                    for i in range(remaining):
                        if len(self.assignments[current_challenge]) < 4:
                            self.assign_student_to_challenge(compatible_students[2+i], current_challenge)
            
            # Actualizar prioridades de los no asignados
            self.update_student_priorities(current_challenge)
        
        # Ahora manejaremos los estudiantes restantes de manera diferente
        self.assign_remaining_students_in_pairs()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [530]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = set(student['Nombre'] for student in self.students)
        self.students_dict = {student['Nombre']: student for student in self.students}
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones de prioridad 1 por desafío."""
        challenge_counts = defaultdict(int)
        for student_name in self.unassigned_students:
            student = self.students_dict[student_name]
            if student['Postulaciones']:  # Si tiene postulaciones pendientes
                # Solo contar la primera prioridad actual
                challenge_counts[student['Postulaciones'][0]] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        # Verificar si el estudiante ya está asignado
        if student['Nombre'] in self.student_assignments:
            return False
            
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo - Estricto máximo de 4
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        career_limit = self.get_career_limit(student['Carrera'])
        if career_counts[student['Carrera']] >= career_limit:
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            if challenge in student['Postulaciones']:
                student['Postulaciones'].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if student['Nombre'] in self.student_assignments:
            return False

        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
            
        if len(self.assignments[challenge]) >= 4:
            return False
            
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.discard(student['Nombre'])
        return True

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def get_compatible_students_for_challenge(self, challenge: str, students: List[dict]) -> List[dict]:
        """
        Obtiene una lista de estudiantes compatibles para un desafío,
        considerando las restricciones de carrera.
        """
        compatible_students = []
        career_counts = defaultdict(int)  # Llevar el conteo de carreras
    
        for student in students:
        # Verificar si agregar este estudiante excedería el límite de su carrera
            if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
                continue
            
        # Si llegamos aquí, el estudiante es compatible
            compatible_students.append(student)
            career_counts[student['Carrera']] += 1
        
        # Si ya tenemos 4 estudiantes compatibles, es suficiente
            if len(compatible_students) >= 4:
                break
    
        return compatible_students

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = self.get_career_limit(career)
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = len(self.student_assignments)
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}, {len(team)}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}, {len(team)}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def assign_remaining_students_in_pairs(self):
        """
        Asigna los estudiantes restantes asegurando que siempre se formen equipos
        de al menos 2 estudiantes y máximo 4.
        """
        while len(self.unassigned_students) >= 2:
            remaining_students = [self.students_dict[name] for name in self.unassigned_students]
            found_pair = False
            
            # Primero intentar completar equipos existentes pequeños
            for challenge, team in self.assignments.items():
                if len(team) < 3:  # Priorizar equipos pequeños
                    career_counts = defaultdict(int)
                    for member in team:
                        career_counts[member['Carrera']] += 1
                    
                    compatible_for_team = []
                    for student in remaining_students:
                        if (career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']) and 
                            self.can_add_student_to_challenge(student, challenge)):
                            compatible_for_team.append(student)
                            
                    if len(compatible_for_team) >= 2:
                        # Intentar agregar dos estudiantes compatibles
                        students_to_add = compatible_for_team[:2]
                        for student in students_to_add:
                            if self.can_add_student_to_challenge(student, challenge):
                                self.assign_student_to_challenge(student, challenge)
                        found_pair = True
                        break
            
            if not found_pair:
                # Intentar agregar a equipos existentes con espacio
                for challenge, team in self.assignments.items():
                    if len(team) < 4:
                        career_counts = defaultdict(int)
                        for member in team:
                            career_counts[member['Carrera']] += 1
                        
                        for student in remaining_students:
                            if (career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']) and 
                                self.can_add_student_to_challenge(student, challenge)):
                                self.assign_student_to_challenge(student, challenge)
                                found_pair = True
                                break
                    if found_pair:
                        break
            
            if not found_pair:
                # Intentar crear un nuevo equipo
                for challenge in [c['Titulo'] for c in self.challenges]:
                    if challenge not in self.assignments:
                        career_counts = defaultdict(int)
                        compatible_students = []
                        
                        for student in remaining_students:
                            if career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']):
                                compatible_students.append(student)
                                career_counts[student['Carrera']] += 1
                                if len(compatible_students) >= 2:
                                    # Crear nuevo equipo con los estudiantes compatibles
                                    for compatible_student in compatible_students[:2]:
                                        self.assign_student_to_challenge(compatible_student, challenge)
                                    found_pair = True
                                    break
                        if found_pair:
                            break
            
            if not found_pair:
                break

        # Manejar estudiantes individuales restantes
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            for challenge, team in self.assignments.items():
                if len(team) >= 2 and len(team) < 4:
                    career_counts = defaultdict(int)
                    for member in team:
                        career_counts[member['Carrera']] += 1
                    
                    if (career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']) and 
                        self.can_add_student_to_challenge(student, challenge)):
                        self.assign_student_to_challenge(student, challenge)
                        break

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes cuando sea posible y respetando máximo 4."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge or len(current_team) >= 4:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            del self.student_assignments[student['Nombre']]
                            self.assign_student_to_challenge(student, challenge)
                            if len(current_team) >= 3:
                                break

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            priority_students = [
                self.students_dict[student_name] for student_name in self.unassigned_students
                if student_name in self.students_dict and
                self.students_dict[student_name]['Postulaciones'] and
                self.students_dict[student_name]['Postulaciones'][0] == current_challenge
            ]
            
            if len(priority_students) >= 2:
                random.shuffle(priority_students)
                
                # Mantener conteo de carreras para este equipo
                career_counts = defaultdict(int)
                team_members = []
                
                # Primero encontrar estudiantes compatibles respetando límites de carrera
                for student in priority_students:
                    if (career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']) and
                        len(team_members) < 4):
                        team_members.append(student)
                        career_counts[student['Carrera']] += 1
                
                # Solo formar equipo si tenemos al menos 2 estudiantes compatibles
                if len(team_members) >= 2:
                    for student in team_members:
                        self.assign_student_to_challenge(student, current_challenge)
            
            # Actualizar prioridades de los no asignados
            self.update_student_priorities(current_challenge)
        
        self.assign_remaining_students_in_pairs()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [546]:
import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

class TeamFormationHeuristic:
    def __init__(self, data):
        self.students = data['estudiantes']
        self.challenges = data['desafios']
        self.careers = data['carreras']
        self.assignments = {}  # Challenge -> List[Student]
        self.student_assignments = {}  # Student_name -> Challenge
        self.unassigned_students = set(student['Nombre'] for student in self.students)
        self.students_dict = {student['Nombre']: student for student in self.students}
        # Mantener preferencias originales intactas
        self.original_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        # Crear copia de trabajo de las preferencias
        self.current_preferences = {
            student['Nombre']: student['Postulaciones'].copy() 
            for student in self.students
        }
        self.assignment_order = []

    def count_applications_per_challenge(self) -> Dict[str, int]:
        """Cuenta las postulaciones de prioridad 1 por desafío."""
        challenge_counts = defaultdict(int)
        for student_name in self.unassigned_students:
            if self.current_preferences[student_name]:  # Si tiene postulaciones pendientes
                # Solo contar la primera prioridad actual
                challenge_counts[self.current_preferences[student_name][0]] += 1
        return challenge_counts
    
    def get_career_limit(self, career: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if career == "Ingeniería Civil Telemática":
            return 3
        return 1

    def can_add_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""
        # Verificar si el estudiante ya está asignado
        if student['Nombre'] in self.student_assignments:
            return False
            
        if challenge not in self.assignments:
            return True
            
        current_team = self.assignments[challenge]
        
        # Verificar límite de equipo - Estricto máximo de 4
        if len(current_team) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        career_counts = defaultdict(int)
        for team_member in current_team:
            career_counts[team_member['Carrera']] += 1
            
        # Verificar límite de carrera
        career_limit = self.get_career_limit(student['Carrera'])
        if career_counts[student['Carrera']] >= career_limit:
            return False
            
        return True

    def update_student_priorities(self, challenge: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for student_name in list(self.unassigned_students):
            if challenge in self.current_preferences[student_name]:
                self.current_preferences[student_name].remove(challenge)

    def assign_student_to_challenge(self, student: dict, challenge: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if student['Nombre'] in self.student_assignments:
            return False

        if challenge not in self.assignments:
            self.assignments[challenge] = []
            self.assignment_order.append(challenge)
            
        if len(self.assignments[challenge]) >= 4:
            return False
            
        self.assignments[challenge].append(student)
        self.student_assignments[student['Nombre']] = challenge
        self.unassigned_students.discard(student['Nombre'])
        return True

    def get_original_preference_number(self, student_name: str, challenge: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        original_preferences = self.original_preferences[student_name]
        if challenge in original_preferences:
            return f"Preferencia #{original_preferences.index(challenge) + 1}"
        return "Fuera de preferencias"

    def get_compatible_students_for_challenge(self, challenge: str, students: List[dict]) -> List[dict]:
        """
        Obtiene una lista de estudiantes compatibles para un desafío,
        considerando las restricciones de carrera.
        """
        compatible_students = []
        career_counts = defaultdict(int)  # Llevar el conteo de carreras
        
        for student in students:
            # Verificar si agregar este estudiante excedería el límite de su carrera
            if career_counts[student['Carrera']] >= self.get_career_limit(student['Carrera']):
                continue
                
            # Si llegamos aquí, el estudiante es compatible
            compatible_students.append(student)
            career_counts[student['Carrera']] += 1
            
            # Si ya tenemos 4 estudiantes compatibles, es suficiente
            if len(compatible_students) >= 4:
                break
        
        return compatible_students

    def print_team_statistics(self):
        """Imprime las estadísticas detalladas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for challenge in self.assignment_order:
            team = self.assignments[challenge]
                
            print(f"\nDesafío: {challenge}")
            print(f"Número de estudiantes: {len(team)}")
            
            career_distribution = defaultdict(int)
            for student in team:
                career_distribution[student['Carrera']] += 1
            
            print("Distribución por carrera:")
            for career, count in career_distribution.items():
                max_allowed = self.get_career_limit(career)
                print(f"- {career}: {count} (máximo permitido: {max_allowed})")
            
            print("Estudiantes en el equipo:")
            for student in team:
                preference = self.get_original_preference_number(student['Nombre'], challenge)
                print(f"- {student['Nombre']} ({student['Carrera']}) ({preference})")

    def validate_assignments(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_assigned_students = len(self.student_assignments)
        if total_assigned_students != len(self.students):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_assigned_students}, Total: {len(self.students)}")
            return False

        for challenge, team in self.assignments.items():
            # Verificar límites de tamaño de equipo
            if len(team) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {challenge}")
                return False
            if len(team) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {challenge}, {len(team)}")
                return False
                
            # Verificar límites por carrera
            career_counts = defaultdict(int)
            for student in team:
                career_counts[student['Carrera']] += 1
                if career_counts[student['Carrera']] > self.get_career_limit(student['Carrera']):
                    print(f"Error: Demasiados estudiantes de {student['Carrera']} en {challenge}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        small_teams = [challenge for challenge, team in self.assignments.items() if len(team) < 3]
        if small_teams:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for challenge in small_teams:
                print(f"- {challenge}")
        
        return True

    def assign_remaining_students_in_pairs(self):
        """
        Asigna los estudiantes restantes asegurando que siempre se formen equipos
        de al menos 2 estudiantes y máximo 4.
        """
        while len(self.unassigned_students) >= 2:
            remaining_students = [self.students_dict[name] for name in self.unassigned_students]
            found_pair = False
            
            # Primero intentar completar equipos existentes pequeños
            for challenge, team in self.assignments.items():
                if len(team) < 3:  # Priorizar equipos pequeños
                    career_counts = defaultdict(int)
                    for member in team:
                        career_counts[member['Carrera']] += 1
                    
                    compatible_for_team = []
                    for student in remaining_students:
                        if (career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']) and 
                            self.can_add_student_to_challenge(student, challenge)):
                            compatible_for_team.append(student)
                            
                    if len(compatible_for_team) >= 2:
                        # Intentar agregar dos estudiantes compatibles
                        students_to_add = compatible_for_team[:2]
                        for student in students_to_add:
                            if self.can_add_student_to_challenge(student, challenge):
                                self.assign_student_to_challenge(student, challenge)
                        found_pair = True
                        break
            
            if not found_pair:
                # Intentar crear un nuevo equipo
                for challenge in [c['Titulo'] for c in self.challenges]:
                    if challenge not in self.assignments:
                        career_counts = defaultdict(int)
                        compatible_students = []
                        
                        for student in remaining_students:
                            if career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']):
                                compatible_students.append(student)
                                career_counts[student['Carrera']] += 1
                                if len(compatible_students) >= 2:
                                    # Crear nuevo equipo con los estudiantes compatibles
                                    for compatible_student in compatible_students[:2]:
                                        self.assign_student_to_challenge(compatible_student, challenge)
                                    found_pair = True
                                    break
                        if found_pair:
                            break
            
            if not found_pair:
                break

        # Manejar estudiantes individuales restantes
        for student_name in list(self.unassigned_students):
            student = self.students_dict[student_name]
            for challenge, team in self.assignments.items():
                if len(team) >= 2 and len(team) < 4:
                    career_counts = defaultdict(int)
                    for member in team:
                        career_counts[member['Carrera']] += 1
                    
                    if (career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']) and 
                        self.can_add_student_to_challenge(student, challenge)):
                        self.assign_student_to_challenge(student, challenge)
                        break

    def complete_teams(self):
        """Completa equipos asegurando mínimo 3 estudiantes cuando sea posible y respetando máximo 4."""
        incomplete_teams = [
            challenge for challenge, team in self.assignments.items()
            if len(team) < 3
        ]
        
        for challenge in incomplete_teams:
            current_team = self.assignments[challenge]
            
            # Intentar mover estudiantes de equipos grandes
            for other_challenge, other_team in self.assignments.items():
                if other_challenge == challenge or len(current_team) >= 4:
                    continue
                if len(other_team) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for student in other_team[:]:
                        if len(current_team) < 3 and self.can_add_student_to_challenge(student, challenge):
                            other_team.remove(student)
                            del self.student_assignments[student['Nombre']]
                            self.assign_student_to_challenge(student, challenge)
                            if len(current_team) >= 3:
                                break

    def form_teams(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Forma los equipos siguiendo la heurística especificada."""
        while self.unassigned_students:
            challenge_counts = self.count_applications_per_challenge()
            if not challenge_counts:
                break
                
            current_challenge = max(challenge_counts.items(), key=lambda x: x[1])[0]
            
            priority_students = [
                self.students_dict[student_name] for student_name in self.unassigned_students
                if student_name in self.students_dict and
                self.current_preferences[student_name] and
                self.current_preferences[student_name][0] == current_challenge
            ]
            
            if len(priority_students) >= 2:
                random.shuffle(priority_students)
                
                # Mantener conteo de carreras para este equipo
                career_counts = defaultdict(int)
                team_members = []
                
                # Primero encontrar estudiantes compatibles respetando límites de carrera
                for student in priority_students:
                    if (career_counts[student['Carrera']] < self.get_career_limit(student['Carrera']) and
                        len(team_members) < 4):
                        team_members.append(student)
                        career_counts[student['Carrera']] += 1
                
                # Solo formar equipo si tenemos al menos 2 estudiantes compatibles
                if len(team_members) >= 2:
                    for student in team_members:
                        self.assign_student_to_challenge(student, current_challenge)
            
            # Actualizar prioridades de los no asignados
            self.update_student_priorities(current_challenge)
        
        self.assign_remaining_students_in_pairs()
        self.complete_teams()
        
        return self.assignments, self.student_assignments

In [556]:
# Cargar datos
with open('body.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Ejecutar el algoritmo
assignments, student_assignments = main(data)


Advertencia: Los siguientes desafíos tienen equipos de 2 integrantes:
- People Analytics - Talento y Desarrollo
Asignaciones válidas completadas

=== Métricas de Asignación ===
Satisfacción promedio: 0.734
Estudiantes en primera prioridad: 41
Estudiantes fuera de preferencias: 10
Desafíos sin equipo: 19
Desviación estándar tamaño equipos: 0.581
Promedio de carreras por equipo: 1.74

Porcentajes:
Primera prioridad: 64.1%
Fuera de preferencias: 15.6%

Estadísticas de asignación:

Desafío: Clasificación de imágenes de mamografía usando Machine Learning
Número de estudiantes: 4
Distribución por carrera:
- Ingeniería Civil Telemática: 3 (máximo permitido: 3)
- Ingeniería Civil Industrial: 1 (máximo permitido: 1)
Estudiantes en el equipo:
- Santiago Lopez (Ingeniería Civil Telemática) (Preferencia #1)
- JUAN PATRICIO ARDILES CASTILLO (Ingeniería Civil Telemática) (Preferencia #1)
- Vicente Andrés Llanos Álvarez (Ingeniería Civil Telemática) (Preferencia #1)
- Valentina Paz Lobos Páez (Ingen